## Project Description

intro text: 

Copenhagen has changed a lot in recent years: the streets are cleaner, the coffee is better, and the prices are much, much higher. For locals mourning the loss of “old Copenhagen,” the newcomers from contrasting Jutland are an easy target.

The long-running “feud” between born-and-bred Copenhageners and newcomers from Jutland shows no sign of cooling off.

But are these newcomers, with revaling  accents really to blame? Who exactly are the Jutlanders, and where do they pop up in the city? What patterns can we spot, and what do they reveal about this quietly invading group?


This project combines two datasets:
- `residents`: place of birth registration and current place of residence.
- `bydel`: GeoJSON boundaries for Copenhagen neighborhoods (bydele).

## Main Question

Which Copenhagen neighborhoods attract residents born in Jutland and Funen, and how does this change over time?

## Possible Sub-Questions

1. Do people from different parts of Jutland/Funen cluster in different Copenhagen neighborhoods?
2. Are there any over-time trend in where newcomers settle?
3. Are some neighborhoods consistently more attractive than others?
4. Are there visible geographic patterns between birth region and destination neighborhood?

## Visualization Ideas

1. **Interactive choropleth map (Folium)**: Color neighborhoods by number of residents from selected birth regions.
2. **Time slider map**: Show how neighborhood counts change by year.
5. **Stacked bar chart by year**: Show neighborhood composition over time.
6. **Heatmap table**: Birth region on one axis, destination neighborhood on the other.
7. **Small multiples (faceted maps)**: One map per year or per birth region for direct comparison.
8. **Net change plot**: Compare growth/decline in each neighborhood across years.

## Suggested Interactive View

Build an interactive map where users can:
- filter by birth region (for example North Jutland, East Jutland, Funen),
- select year or year range,
- see both neighborhood color intensity and exact counts in tooltips/popups.

In [14]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
from pathlib import Path
import json
import folium


In [15]:
data_dir = Path("data")
geo_path = data_dir / "bydel.csv"

with geo_path.open("r", encoding="utf-8") as f:
    geo = json.load(f)


def ring_centroid_and_area(ring):
    # Shoelace centroid + signed area; ring is [lon, lat].
    if len(ring) < 3:
        return (ring[0][0], ring[0][1], 0.0) if ring else (0.0, 0.0, 0.0)

    twice_area = 0.0
    cx = 0.0
    cy = 0.0

    for i in range(len(ring) - 1):
        x1, y1 = ring[i]
        x2, y2 = ring[i + 1]
        cross = x1 * y2 - x2 * y1
        twice_area += cross
        cx += (x1 + x2) * cross
        cy += (y1 + y2) * cross

    if twice_area == 0:
        xs = [p[0] for p in ring]
        ys = [p[1] for p in ring]
        return (sum(xs) / len(xs), sum(ys) / len(ys), 0.0)

    return (cx / (3 * twice_area), cy / (3 * twice_area), abs(twice_area) / 2.0)


# Build Folium base map with neighborhood outlines and labels
base_map = folium.Map(location=[55.68, 12.56], zoom_start=11, tiles="cartodbpositron")

folium.GeoJson(
    geo,
    style_function=lambda _x: {
        "color": "#374151",
        "weight": 1.5,
        "fillColor": "#93c5fd",
        "fillOpacity": 0.25,
    },
).add_to(base_map)

for feat in geo["features"]:
    name = feat["properties"].get("navn", "")
    geom = feat["geometry"]
    geom_type = geom["type"]
    rings = []
    if geom_type == "Polygon":
        rings = [geom["coordinates"][0]]
    elif geom_type == "MultiPolygon":
        rings = [poly[0] for poly in geom["coordinates"]]

    if not rings:
        continue

    # Pick the ring with the largest area for label placement
    best = max(rings, key=lambda r: ring_centroid_and_area(r)[2])
    lon, lat, _ = ring_centroid_and_area(best)

    folium.Marker(
        location=[lat, lon],
        icon=folium.DivIcon(
            html=f'<div style="font-size:9px;font-weight:bold;color:#1e3a5f;'
                 f'white-space:nowrap;text-shadow:1px 1px 2px #fff,-1px -1px 2px #fff;">'
                 f'{name}</div>',
            icon_size=(120, 20),
            icon_anchor=(60, 10),
        ),
    ).add_to(base_map)

#base_map


In [22]:
import copy
import json
from pathlib import Path

import pandas as pd
import folium
import ipywidgets as widgets
from IPython.display import clear_output, display

# Clear this cell's previous rendered app before creating a new one.
clear_output(wait=True)

# Close previous widget instances from earlier runs to avoid duplicate map views.
for _wname in ["region_selector", "year_slider", "sex_dropdown", "out", "controls", "app"]:
    if _wname in globals():
        try:
            globals()[_wname].close()
        except Exception:
            pass

# Load residents table exported from Statistikbanken-style layout
residents_path = Path("data") / "residents.xlsx"
raw = pd.read_excel(residents_path, sheet_name=0, header=None)

# Detect the row that contains year labels (1977..2026 etc.)
year_count_by_row = raw.apply(
    lambda r: pd.to_numeric(r.iloc[4:], errors="coerce").between(1900, 2100).sum(),
    axis=1,
)
year_row_idx = int(year_count_by_row.idxmax())

years = [
    int(v)
    for v in pd.to_numeric(raw.iloc[year_row_idx, 4:], errors="coerce").dropna().tolist()
]

cols = ["age_group", "sex", "neighborhood_raw", "birth_region"] + years
df = raw.iloc[year_row_idx + 1 :, : len(cols)].copy()
df.columns = cols

# Fill merged header-like cells in first dimensions
for c in ["age_group", "sex", "neighborhood_raw", "birth_region"]:
    df[c] = df[c].ffill()

# Keep total age rows and numeric year values
df = df[df["age_group"] == "Alder i alt"].copy()
for y in years:
    df[y] = pd.to_numeric(df[y], errors="coerce").fillna(0)

# Harmonize neighborhood names between residents table and GeoJSON
name_map = {
    "Vesterbro/Kongens Enghave": "Vesterbro-Kongens Enghave",
}
df["navn"] = (
    df["neighborhood_raw"]
    .astype(str)
    .str.replace("Bydel - ", "", regex=False)
    .replace(name_map)
)

regions = sorted(df["birth_region"].dropna().astype(str).unique().tolist())
sexes = sorted(df["sex"].dropna().astype(str).unique().tolist())

region_selector = widgets.SelectMultiple(
    options=regions,
    value=tuple(regions[:3]) if len(regions) >= 3 else tuple(regions),
    description="Birth region",
    rows=10,
    layout=widgets.Layout(width="420px"),
)

year_slider = widgets.IntSlider(
    value=max(years),
    min=min(years),
    max=max(years),
    step=1,
    description="Year",
    continuous_update=False,
)

sex_dropdown = widgets.Dropdown(
    options=["All"] + sexes,
    value="All",
    description="Sex",
)

out = widgets.Output()


def draw_map(_=None):
    with out:
        out.clear_output(wait=True)

        selected_regions = list(region_selector.value)
        year = year_slider.value
        sex_value = sex_dropdown.value

        data = df.copy()
        if selected_regions:
            data = data[data["birth_region"].astype(str).isin(selected_regions)]
        if sex_value != "All":
            data = data[data["sex"].astype(str) == sex_value]

        agg = data.groupby("navn", as_index=False)[year].sum().rename(columns={year: "value"})

        value_lookup = dict(zip(agg["navn"], agg["value"]))
        geo_plot = copy.deepcopy(geo)
        for feat in geo_plot["features"]:
            n = feat["properties"].get("navn", "Unknown")
            feat["properties"]["value"] = float(value_lookup.get(n, 0))

        m = folium.Map(location=[55.68, 12.56], zoom_start=12.2, tiles="cartodbpositron")

        folium.Choropleth(
            geo_data=geo_plot,
            data=agg,
            columns=["navn", "value"],
            key_on="feature.properties.navn",
            fill_color="YlOrRd",
            fill_opacity=0.75,
            line_opacity=0.35,
            nan_fill_color="#043A97",
            legend_name=f"Residents in {year}",
            name="Residents",
        ).add_to(m)

        folium.GeoJson(
            geo_plot,
            style_function=lambda _x: {"color": "#111827", "weight": 1, "fillOpacity": 0},
            tooltip=folium.GeoJsonTooltip(
                fields=["navn", "value"],
                aliases=["Neighborhood", "Residents"],
                localize=True,
                sticky=False,
            ),
        ).add_to(m)

        # Add neighborhood name labels
        for feat in geo_plot["features"]:
            name = feat["properties"].get("navn", "")
            geom = feat["geometry"]
            rings = []
            if geom["type"] == "Polygon":
                rings = [geom["coordinates"][0]]
            elif geom["type"] == "MultiPolygon":
                rings = [poly[0] for poly in geom["coordinates"]]
            if not rings:
                continue
            best = max(rings, key=lambda r: ring_centroid_and_area(r)[2])
            lon, lat, _ = ring_centroid_and_area(best)
            folium.Marker(
                location=[lat, lon],
                icon=folium.DivIcon(
                    html=f'<div style="font-size:9px;font-weight:bold;color:#1e3a5f;'
                         f'white-space:nowrap;text-shadow:1px 1px 2px #fff,-1px -1px 2px #fff;">'
                         f'{name}</div>',
                    icon_size=(120, 20),
                    icon_anchor=(60, 10),
                ),
            ).add_to(m)

        display(m)
        m.save("map.html")  # <-- Add this here!
        print("Saved interactive map with Jylland data as map.html")


region_selector.observe(draw_map, names="value")
year_slider.observe(draw_map, names="value")
sex_dropdown.observe(draw_map, names="value")

controls = widgets.VBox([
    widgets.HTML("<b>Interactive choropleth: residents by birth region</b>"),
    sex_dropdown,
    year_slider,
    region_selector,
])

app = widgets.VBox([controls, out])
display(app)



In [17]:
# After drawing your map, save it as interactive HTML for your website
base_map.save("map.html")
print("Saved interactive map as map.html")

Saved interactive map as map.html


In [18]:
import plotly.express as px
import numpy as np

# Filter to all birth regions containing "jylland" (case-insensitive)
jylland_mask = df["birth_region"].astype(str).str.contains("jylland", case=False, na=False)
df_jyl = df[jylland_mask].copy()

jylland_regions = sorted(df_jyl["birth_region"].astype(str).unique().tolist())
print(f"Jylland birth regions found ({len(jylland_regions)}):")
for r in jylland_regions:
    print(" ", r)


Jylland birth regions found (4):
  Nordjylland
  Sydjylland
  Vestjylland
  Østjylland


In [19]:
latest_year = max(years)

# Aggregate over sex, pivot: birth_region × neighborhood
pivot = (
    df_jyl.groupby(["birth_region", "navn"])[latest_year]
    .sum()
    .unstack(fill_value=0)
)

# Normalize by row so each birth region's row sums to 100 %
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

fig_heat = px.imshow(
    pivot_pct,
    labels=dict(x="Neighborhood", y="Birth region", color="% of birth region"),
    title=f"How each Jylland birth region distributes across Copenhagen neighborhoods ({latest_year})",
    aspect="auto",
    color_continuous_scale="YlOrRd",
    text_auto=".1f",
)
fig_heat.update_layout(
    height=max(400, 50 * len(jylland_regions)),
    xaxis_tickangle=-45,
    font=dict(size=11),
)
fig_heat.show()


In [23]:
# Total Jylland residents per neighborhood per year (all years, all Jylland regions summed)
jyl_by_year = (
    df_jyl.groupby("navn")[years]
    .sum()
    .T  # rows = years, columns = neighborhoods
    .reset_index()
    .rename(columns={"index": "year"})
)
jyl_long = jyl_by_year.melt(id_vars="year", var_name="Neighborhood", value_name="Residents")
jyl_long["year"] = jyl_long["year"].astype(int)

# Drop non-geographic catch-all area
jyl_long = jyl_long[~jyl_long["Neighborhood"].str.contains("uden for", case=False, na=False)]

fig_line = px.line(
    jyl_long,
    x="year",
    y="Residents",
    color="Neighborhood",
    title="Number of Jutlanders by Copenhagen Area over Time",
    labels={"year": "Year", "Residents": "Number of residents born in Jylland"},
    markers=False,
)
fig_line.update_layout(
    height=550,
    xaxis=dict(dtick=5),
    legend=dict(title="Neighborhood", font=dict(size=10)),
    hovermode="x unified",
)
fig_line.show()
fig_line.write_html("time_plot.html")



When we map the density of Jutlanders across Copenhagen’s neighborhoods over time, a familiar pattern emerges: they have excellent timing.

The timeline stretches back to the mid-1970s, when the numbers were already high. Drawn in by jobs and education, people from rural Denmark arrived in the capital and settled where housing was cheap and plentiful. The data shows the highest concentrations in Nørrebro and Østerbro—industrial, working-class districts that were a bit rough around the edges, but affordable and, crucially, available.

Then came the late 1980s and early 1990s, and something curious happened: they left. Or at least, many of them did. Across neighborhoods, the number of Jutlanders declines steadily from the 1980s through the mid-90s and into the early 2000s. As industry faded, jobs disappeared, and these areas struggled with unemployment and social challenges, the charm—unsurprisingly—wore off. The affordable city was still there, just less appealing.

But the story doesn’t end there. From around 2000 to 2010, the curve shoots up again. The biggest surge appears in the very neighborhoods once written off—Nørrebro, Østerbro, Amager, and especially Vesterbro. By then, they had been “sanitized,” renovated, and carefully rebranded. Cafés replaced factories, rents climbed accordingly, and suddenly Nørrebro was no longer a compromise—it was a lifestyle.

And, right on cue, the Jutlanders returned.